In [ ]:
from python_code.channel.modulator import BPSKModulator
import torch
import numpy as np
import math

def create_transition_table(n_states: int) -> np.ndarray:
    """
    creates transition table of size [n_states,2]
    previous state of state i and input bit b is the state in cell [i,b]
    """
    transition_table = np.concatenate([np.arange(n_states), np.arange(n_states)]).reshape(n_states, 2)
    return transition_table


device = "cpu"
block_length = 10
memory_length = 2
n_states = 2 ** memory_length
transmission_length = block_length
batch_size = 1

h = np.array([[1.0, 0.5]])  # Example channel coefficients
y = torch.zeros(batch_size, transmission_length)

c = np.array([[1,1,2,2,1,2,2,1,1,1]])-1
padded_c = np.concatenate([c, np.zeros([c.shape[0], memory_length])], axis=1)
s = 1 - 2 * padded_c
blockwise_s = np.concatenate([s[:, i:-memory_length + i] for i in range(memory_length)], axis=0)
conv = np.dot(h[:, ::-1], blockwise_s)
[row, col] = conv.shape
y[0, :] = torch.tensor(conv)

transition_table_array = create_transition_table(n_states)
transition_table = torch.Tensor(transition_table_array).to(device)

snr = 0
all_states_decimal = np.arange(n_states).astype(np.uint8).reshape(-1, 1)
all_states_binary = np.unpackbits(all_states_decimal, axis=1).astype(int)
all_states_symbols = BPSKModulator.modulate(all_states_binary[:, -memory_length:])
state_priors = np.dot(all_states_symbols, h[:,::-1].T)
state_priors = torch.Tensor(state_priors).to(device)

priors = y.unsqueeze(dim=2) - state_priors.T.repeat(
    repeats=[y.shape[0] // state_priors.shape[1], 1]).unsqueeze(
    dim=1)
# to llr representation
sigma = 1 / 10 ** (-snr / 10)
priors = priors ** 2 / (2 * sigma ** 2) + math.log(math.sqrt(2 * math.pi) * sigma)

In [21]:

transition_table_array = create_transition_table(n_states)
transition_table = torch.Tensor(transition_table_array).to(device)

in_prob = torch.zeros([y.shape[0], n_states]).to(device)
decoded_word = torch.zeros(y.shape).to(device)
# for i in range(transmission_length):
i = 0
def viterbi_step(i, in_prob):
    # get the lsb of the state
    decoded_word[:, i] = torch.argmin(in_prob, dim=1) % 2
    # run one Viterbi stage (acs block)
    llrs = priors[:, i]
    transition_ind = transition_table.reshape(-1).repeat(in_prob.size(0)).long()
    batches_ind = torch.arange(in_prob.size(0)).repeat_interleave(2 * n_states)
    trellis = (in_prob + llrs)[batches_ind, transition_ind]
    reshaped_trellis = trellis.reshape(-1, n_states, 2)
    out_prob, _ = torch.min(reshaped_trellis, dim=2)
    print(f"========= Viterbi Stage {i} =========")
    print("decoded_word:", decoded_word)
    print("in_prob:", in_prob[0])
    print("llrs:", llrs[0])
    print("trellis:", reshaped_trellis[0])
    print("out_prob:", out_prob[0])
    # update in-probabilities for next layer
    in_prob = out_prob

    return in_prob

for i in range(transmission_length):
    in_prob = viterbi_step(i, in_prob)

========= Viterbi Stage 0 =========
decoded_word: tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])
in_prob: tensor([0., 0., 0., 0.])
llrs: tensor([3.2215, 3.2415, 3.2265, 3.2665])
trellis: tensor([[3.2215, 3.2415],
        [3.2265, 3.2665],
        [3.2215, 3.2415],
        [3.2265, 3.2665]])
out_prob: tensor([3.2215, 3.2265, 3.2215, 3.2265])
========= Viterbi Stage 1 =========
decoded_word: tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])
in_prob: tensor([3.2215, 3.2265, 3.2215, 3.2265])
llrs: tensor([3.2415, 3.2215, 3.2265, 3.2265])
trellis: tensor([[6.4630, 6.4480],
        [6.4480, 6.4530],
        [6.4630, 6.4480],
        [6.4480, 6.4530]])
out_prob: tensor([6.4480, 6.4480, 6.4480, 6.4480])
========= Viterbi Stage 2 =========
decoded_word: tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])
in_prob: tensor([6.4480, 6.4480, 6.4480, 6.4480])
llrs: tensor([3.2665, 3.2265, 3.2415, 3.2215])
trellis: tensor([[9.7146, 9.6746],
        [9.6896, 9.6696],
        [9.7146, 9.6746],
     

In [22]:
c

array([[0, 0, 1, 1, 0, 1, 1, 0, 0, 0]])

In [23]:
decoded_word

tensor([[0., 0., 0., 1., 0., 0., 1., 0., 0., 0.]])